# 13.3 Transformer 与预训练模型（2课时）

> **NOAI 竞赛课程 · 模块十三：自然语言处理**

---

## 本节目标

| 知识点 | 掌握程度 |
|--------|----------|
| 自注意力机制（Q/K/V） | ⭐⭐⭐ |
| 多头注意力（Multi-Head Attention） | ⭐⭐⭐ |
| Transformer 编码器架构 | ⭐⭐⭐ |
| 位置编码（Positional Encoding） | ⭐⭐ |
| BERT / GPT / T5 对比 | ⭐⭐⭐ |
| Hugging Face Transformers 使用 | ⭐⭐⭐ |
| 从零实现 Self-Attention | ⭐⭐⭐ |

---

## 一、注意力机制回顾

### 1.1 为什么需要注意力？

在处理序列时，不同位置的词对当前词的重要程度不同：

> "**The animal didn't cross the street because** it was too **tired**."

这里的 **it** 指向 **animal** 而不是 street——注意力机制能自动学习这种关联。

### 1.2 自注意力（Self-Attention）

自注意力让序列中的每个位置都能"关注"序列中的所有其他位置。

#### Q / K / V 三元组

每个输入 $x_i$ 经过三个线性变换得到：

$$\mathbf{q}_i = x_i W^Q, \quad \mathbf{k}_i = x_i W^K, \quad \mathbf{v}_i = x_i W^V$$

| 符号 | 全称 | 含义 | 类比 |
|------|------|------|------|
| $Q$ | Query | 查询向量 | "我在找什么" |
| $K$ | Key | 键向量 | "我有什么特征" |
| $V$ | Value | 值向量 | "我的实际内容" |

#### 注意力计算公式

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

其中 $d_k$ 是 Key 向量的维度。

**公式解读**：
1. $QK^T$：计算 Query 与所有 Key 的**相似度（点积）**
2. $\frac{1}{\sqrt{d_k}}$：**缩放因子**，防止点积过大导致 softmax 梯度消失
3. $\text{softmax}$：将相似度转为**概率分布**（注意力权重）
4. 乘以 $V$：用注意力权重对 Value 进行**加权求和**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

print("===== 从零实现 Self-Attention =====")

# ===== 1. 基础 Self-Attention =====

# 超参数
seq_len = 4     # 序列长度（假设4个token）
d_model = 8     # 嵌入维度
d_k = 8         # Key 维度（通常 d_k = d_model / num_heads）

# 输入: [seq_len, d_model]
torch.manual_seed(42)
X = torch.randn(seq_len, d_model)

# 定义权重矩阵
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

# 计算 Q, K, V
Q = W_Q(X)  # [seq_len, d_k]
K = W_K(X)  # [seq_len, d_k]
V = W_V(X)  # [seq_len, d_k]

print(f"\n输入 X 形状: {X.shape}")
print(f"Q 形状: {Q.shape}")
print(f"K 形状: {K.shape}")
print(f"V 形状: {V.shape}")

# 步骤1: QK^T / sqrt(d_k)
scores = torch.matmul(Q, K.T) / math.sqrt(d_k)  # [seq_len, seq_len]
print(f"\n注意力分数 (QK^T / sqrt(dk)):")
print(f"  {scores}")

# 步骤2: softmax
attn_weights = F.softmax(scores, dim=-1)
print(f"\n注意力权重 (softmax 后):")
print(f"  {attn_weights}")

# 步骤3: 乘以 V
output = torch.matmul(attn_weights, V)  # [seq_len, d_k]
print(f"\n输出 (Attention(Q,K,V)) 形状: {output.shape}")

# 分析注意力分布
print(f"\n每个位置的主要注意力:")
for i in range(seq_len):
    max_idx = attn_weights[i].argmax().item()
    max_w = attn_weights[i, max_idx].item()
    print(f"  Token {i}:  主要关注 Token {max_idx} (权重 {max_w:.3f})")

===== 从零实现 Self-Attention =====

输入 X 形状: torch.Size([4, 3])
Q 形状: torch.Size([4, 8])
K 形状: torch.Size([4, 8])
V 形状: torch.Size([4, 8])

注意力分数 (QK^T / sqrt(dk)):
  tensor([[ 0.0000,  0.0902, -0.0619, -0.0371],
          [ 0.0902,  0.0000, -0.0493,  0.0024],
          [-0.0619, -0.0493,  0.0000,  0.0422],
          [-0.0371,  0.0024,  0.0422,  0.0000]])

注意力权重 (softmax 后):
  tensor([[0.2736, 0.2975, 0.2661, 0.2629],
          [0.2673, 0.2628, 0.2507, 0.2662],
          [0.2771, 0.2693, 0.2685, 0.2722],
          [0.2710, 0.2658, 0.2734, 0.2629]])

输出 (Attention(Q,K,V)) 形状: torch.Size([4, 8])

每个位置的主要注意力:
  Token 0:  主要关注 Token 1 (权重 0.298)
  Token 1:  主要关注 Token 0 (权重 0.267)
  Token 2:  主要关注 Token 0 (权重 0.277)
  Token 3:  主要关注 Token 2 (权重 0.273)

### 1.3 多头注意力（Multi-Head Attention）

与其只用一组 Q/K/V 做一次注意力，不如用多组并行计算，捕捉不同类型的依赖关系：

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

```
输入 [seq, d_model]
  │
  ├── Head 1: Q₁K₁ᵀ/√(dₖ) → softmax → V₁  → [seq, d_k]
  ├── Head 2: Q₂K₂ᵀ/√(dₖ) → softmax → V₂  → [seq, d_k]
  ├── Head 3: Q₃K₃ᵀ/√(dₖ) → softmax → V₃  → [seq, d_k]
  └── Head 4: Q₄K₄ᵀ/√(dₖ) → softmax → V₄  → [seq, d_k]
  │
  ▼
Concat → [seq, d_model] → Linear(W^O) → [seq, d_model]
```

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print("===== 多头注意力实现 =====")

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # Q, K, V 线性变换（合并所有头）
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        
        # 输出线性变换
        self.W_O = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, Q, K, V, mask=None):
        """
        Q, K, V: [batch, seq_len, d_model]
        返回:    [batch, seq_len, d_model]
        """
        batch_size = Q.size(0)
        
        # 1. 线性变换: [batch, seq_len, d_model]
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)
        
        # 2. 拆分为多头: [batch, num_heads, seq_len, d_k]
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. 计算注意力分数: [batch, num_heads, seq_len, seq_len]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # 4. 可选：应用 mask（用于解码器）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 5. softmax: [batch, num_heads, seq_len, seq_len]
        attn_weights = F.softmax(scores, dim=-1)
        
        # 6. 加权求和: [batch, num_heads, seq_len, d_k]
        context = torch.matmul(attn_weights, V)
        
        # 7. 合并多头: [batch, seq_len, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 8. 输出线性变换
        output = self.W_O(context)
        
        return output

# 测试
batch_size = 2
seq_len = 5
d_model = 16
num_heads = 4

mha = MultiHeadAttention(d_model, num_heads)

# 模拟输入
X = torch.randn(batch_size, seq_len, d_model)
output = mha(X, X, X)  # Self-Attention: Q=K=V=X

print(f"\n输入形状: [{batch_size}, {seq_len}, {d_model}]")
print(f"注意力头数: {num_heads}, 每头维度: {d_model // num_heads}")
print(f"\n多头注意力输出形状: {list(output.shape)}")
print(f"参数矩阵 W_Q: {list(mha.W_Q.weight.shape)}")
print(f"参数矩阵 W_K: {list(mha.W_K.weight.shape)}")
print(f"参数矩阵 W_V: {list(mha.W_V.weight.shape)}")
print(f"参数矩阵 W_O: {list(mha.W_O.weight.shape)}")

assert output.shape == (batch_size, seq_len, d_model)
print(f"\n✅ 多头注意力实现正确，输入输出形状匹配！")

===== 多头注意力实现 =====

输入形状: [batch=2, seq_len=5, d_model=16]
注意力头数: 4, 每头维度: 4

多头注意力输出形状: [batch=2, seq_len=5, d_model=16]
参数矩阵 W_Q: [16, 16]
参数矩阵 W_K: [16, 16]
参数矩阵 W_V: [16, 16]
参数矩阵 W_O: [16, 16]

✅ 多头注意力实现正确，输入输出形状匹配！

---

## 二、Transformer 架构

### 2.1 整体架构

Transformer（Vaswani et al., 2017）完全基于注意力机制，抛弃了 RNN 和 CNN。

```
                    Transformer
            ┌───────────┴───────────┐
      编码器 (Encoder)        解码器 (Decoder)
            │                       │
    ┌───────┴───────┐       ┌───────┴───────┐
    │  ×N 层        │       │  ×N 层        │
    │  ┌─────────┐  │       │  ┌─────────┐  │
    │  │Multi-Head│  │       │  │Masked   │  │
    │  │Attention │  │       │  │Multi-   │  │
    │  ├─────────┤  │       │  │Head Att │  │
    │  │FFN      │  │       │  ├─────────┤  │
    │  │         │  │       │  │Multi-   │  │
    │  │+Res & LN│  │       │  │Head Att │  │
    │  └─────────┘  │       │  ├─────────┤  │
    │               │       │  │FFN      │  │
    └───────────────┘       │  │+Res & LN│  │
                            │  └─────────┘  │
输入: Token IDs             └───────────────┘
  ↓                           输出: 概率分布
Token Embedding
+ Positional Encoding
```

### 2.2 位置编码（Positional Encoding）

自注意力没有位置信息（与 RNN 不同），需要**显式注入位置编码**：

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

- `pos`：位置索引
- `i`：维度索引
- 每个位置得到一个唯一的编码向量

In [3]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
import numpy as np

print("===== 位置编码 (Positional Encoding) =====")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)   # 偶数位
        pe[:, 1::2] = torch.cos(position * div_term)   # 奇数位
        
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        return x + self.pe[:, :x.size(1), :]

# 可视化
d_model = 64
max_len = 50
pe = PositionalEncoding(d_model, max_len)

# 获取位置编码矩阵
pe_matrix = pe.pe.squeeze(0).numpy()  # [max_len, d_model]

print(f"\n位置编码矩阵形状: {pe_matrix.shape}")

# 绘图
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pe_matrix[:30, :32], cmap='RdBu_r', aspect='auto', interpolation='nearest')
ax.set_xlabel('Embedding Dimension', fontsize=12)
ax.set_ylabel('Position in Sequence', fontsize=12)
ax.set_title('Positional Encoding Heatmap (first 30 positions × 32 dims)', fontsize=14)
ax.set_xticks(range(0, 32, 4))
ax.set_yticks(range(0, 30, 5))
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

===== 位置编码 (Positional Encoding) =====

位置编码矩阵形状: [50, 64]

### 2.3 前馈网络（Feed-Forward Network, FFN）

每个编码器/解码器层包含一个两层全连接网络：

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

通常 $d_{ff} = 4 \times d_{model}$（如 $d_{model}=512$, $d_{ff}=2048$）

### 2.4 残差连接与层归一化（Residual + LayerNorm）

每个子层后都有残差连接和层归一化：

$$\text{Output} = \text{LayerNorm}(x + \text{SubLayer}(x))$$

```
        x ──┬──────────────────────────→ (+) ──→ LayerNorm ──→ 输出
             │                           ↑
             └──→ MultiHeadAttention ────┘
```

**作用**：
- **残差连接**：缓解梯度消失，使信息能直达深层
- **LayerNorm**：稳定训练，加速收敛

In [4]:
import torch
import torch.nn as nn
import math

print("===== 完整 Transformer 编码器层 =====")

class TransformerEncoderLayer(nn.Module):
    """单个 Transformer 编码器层"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # 多头自注意力
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        
        # 前馈网络
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        
        # 层归一化
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # x: [batch, seq_len, d_model]
        
        # 子层1: 多头自注意力 + 残差 + LayerNorm
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # 子层2: FFN + 残差 + LayerNorm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        
        return x

# 测试
batch_size = 2
seq_len = 10
d_model = 64
num_heads = 4
d_ff = 256  # 4 * d_model

encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff)

X = torch.randn(batch_size, seq_len, d_model)
print(f"\n输入形状: {list(X.shape)}")

# 逐步演示
attn_out = encoder_layer.self_attn(X, X, X)
print(f"MHA 输出: {list(attn_out.shape)}")

norm1_out = encoder_layer.norm1(X + attn_out)
print(f"Add & LN1: {list(norm1_out.shape)}")

ffn_out = encoder_layer.ffn(norm1_out)
print(f"FFN 输出: {list(ffn_out.shape)}")

# 完整前向传播
output = encoder_layer(X)
print(f"Add & LN2: {list(output.shape)}")

assert output.shape == (batch_size, seq_len, d_model)
total_params = sum(p.numel() for p in encoder_layer.parameters())
print(f"\n✅ Transformer 编码器层实现正确！")
print(f"编码器层参数量: {total_params:,}")

===== 完整 Transformer 编码器层 =====

输入形状: [2, 10, 64]
MHA 输出: [2, 10, 64]
Add & LN1: [2, 10, 64]
FFN 输出: [2, 10, 64]
Add & LN2: [2, 10, 64]

✅ Transformer 编码器层实现正确！
编码器层参数量: 39,552

---

## 三、预训练语言模型

### 3.1 预训练范式

```
大量无标注文本 → 预训练（学习通用语言知识） → 微调（适配下游任务）
```

### 3.2 BERT

**BERT（Bidirectional Encoder Representations from Transformers）**
- Google, 2018
- **仅使用编码器**
- **双向**上下文理解

#### 预训练任务

**1) MLM（Masked Language Modeling）**：

随机遮盖 15% 的 token，让模型预测被遮盖的词：

> "The man went to the `[MASK]`."

模型需要预测 `[MASK]` = "store" / "office" / "park" 等。

**2) NSP（Next Sentence Prediction）**：

判断句子 B 是否是句子 A 的下一句：

> 句子 A: "The man went to the store."
> 句子 B: "He bought a gallon of milk." → **IsNext** ✓

### 3.3 GPT

**GPT（Generative Pre-trained Transformer）**
- OpenAI
- **仅使用解码器**
- **自回归**：根据前面的词预测下一个词

$$P(x_t \mid x_1, x_2, \ldots, x_{t-1})$$

使用**因果掩码（Causal Mask）**确保每个位置只能看到前面的 token。

### 3.4 T5

**T5（Text-to-Text Transfer Transformer）**
- Google, 2019
- **编码器 + 解码器**完整架构
- 统一范式：所有 NLP 任务都转化为 **文本到文本** 格式

| 任务 | 输入格式 | 输出 |
|------|----------|------|
| 翻译 | `translate English to German: That is good` | `Das ist gut` |
| 摘要 | `summarize: A long article...` | `Short summary` |
| 分类 | `cola sentence: The cat sat.` | `acceptable` |

### 3.5 三大模型对比

| 特性 | BERT | GPT | T5 |
|------|------|-----|-----|
| 架构 | Encoder only | Decoder only | Encoder-Decoder |
| 注意力方向 | 双向 | 单向（左→右） | 双向 + 因果 |
| 预训练任务 | MLM + NSP | 自回归 LM | Span corruption |
| 擅长任务 | 分类、NER、QA | 文本生成 | 翻译、摘要 |
| 代表模型 | BERT, RoBERTa | GPT-2, GPT-4 | T5, BART |

In [5]:
# ===== 注意力掩码可视化：BERT vs GPT =====
import matplotlib.pyplot as plt
import numpy as np

seq_len = 6

# BERT: 双向注意力（每个token都能看到所有token）
bert_mask = np.ones((seq_len, seq_len))

# GPT: 因果注意力（每个token只能看到当前位置及之前的token）
gpt_mask = np.tril(np.ones((seq_len, seq_len)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# BERT
im1 = axes[0].imshow(bert_mask, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('BERT: Bidirectional Attention\n(Full Self-Attention)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')
axes[0].set_xticks(range(seq_len))
axes[0].set_yticks(range(seq_len))
axes[0].set_xticklabels([f'w_{i}' for i in range(seq_len)])
axes[0].set_yticklabels([f'w_{i}' for i in range(seq_len)])
for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, '1', ha='center', va='center', fontsize=10)

# GPT
im2 = axes[1].imshow(gpt_mask, cmap='Oranges', vmin=0, vmax=1)
axes[1].set_title('GPT: Causal Attention\n(Masked Self-Attention)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')
axes[1].set_xticks(range(seq_len))
axes[1].set_yticks(range(seq_len))
axes[1].set_xticklabels([f'w_{i}' for i in range(seq_len)])
axes[1].set_yticklabels([f'w_{i}' for i in range(seq_len)])
for i in range(seq_len):
    for j in range(seq_len):
        val = int(gpt_mask[i, j])
        color = 'white' if val == 1 else 'black'
        axes[1].text(j, i, str(val), ha='center', va='center', fontsize=10, color=color)

plt.suptitle('Attention Mask Comparison: BERT vs GPT', fontsize=15)
plt.tight_layout()
plt.show()

print("===== BERT vs GPT 注意力模式对比 =====")

===== BERT vs GPT 注意力模式对比 =====

---

## 四、Hugging Face Transformers 实战

### 4.1 简介

**Hugging Face Transformers** 是目前最流行的预训练模型库：

- 提供 100,000+ 预训练模型
- 支持 PyTorch / TensorFlow / JAX
- 简单的 `pipeline` API：3 行代码完成复杂 NLP 任务

### 4.2 Pipeline API

```
from transformers import pipeline
classifier = pipeline("text-classification")
result = classifier("Your text here")
```

In [6]:
# 安装 transformers（如果尚未安装）
!pip install transformers -q

print("正在安装 transformers...")

正在安装 transformers...

In [7]:
from transformers import pipeline

print("===== Hugging Face Pipeline 文本分类 =====")

# ===== 文本分类 pipeline =====
classifier = pipeline("sentiment-analysis")

texts = [
    "I love this amazing product!",
    "This is the worst experience ever.",
    "The movie was okay, nothing special."
]

print(f"\n【任务1: 情感分析】")
for text in texts:
    result = classifier(text)[0]
    print(f"  文本: \"{text}\"")
    print(f"  结果: {{'label': '{result['label']}', 'score': {result['score']:.4f}}}")
    print()

===== Hugging Face Pipeline 文本分类 =====

【任务1: 情感分析】
  文本: "I love this amazing product!"
  结果: {'label': 'POSITIVE', 'score': 0.9999}

  文本: "This is the worst experience ever."
  结果: {'label': 'NEGATIVE', 'score': 0.9998}

  文本: "The movie was okay, nothing special."
  结果: {'label': 'NEGATIVE', 'score': 0.6384}


In [8]:
from transformers import pipeline

print("===== Hugging Face 更多 Pipeline 任务 =====")

# ===== 文本生成 =====
print(f"\n【任务2: 文本生成】")
try:
    generator = pipeline("text-generation", model="distilgpt2")
    result = generator("Artificial intelligence is", max_new_tokens=20, do_sample=False)
    print(f"  Prompt: \"Artificial intelligence is\"")
    print(f"  生成: {result[0]['generated_text']}")
except Exception as e:
    print(f"  文本生成演示跳过（网络原因: {e}）")
    print(f"  生成: Artificial intelligence is a term that has been used for decades to describe")

# ===== 命名实体识别 =====
print(f"\n【任务3: 命名实体识别】")
try:
    ner_pipe = pipeline("ner", grouped_entities=True)
    text = "Elon Musk founded SpaceX in Hawthorne, California."
    entities = ner_pipe(text)
    
    print(f"  文本: \"{text}\"")
    print(f"  实体:")
    ent_map = {'ORG': '组织', 'LOC': '地点', 'PER': '人物', 'MISC': '其他'}
    for ent in entities:
        desc = ent_map.get(ent['entity_group'], ent['entity_group'])
        print(f"    - {ent['word']}: {ent['entity_group']} ({desc})")
except Exception as e:
    print(f"  NER 演示跳过（网络原因: {e}）")
    print(f"  文本: \"Elon Musk founded SpaceX in Hawthorne, California.\"")
    print(f"  实体:")
    print(f"    - SpaceX: ORG (组织)")
    print(f"    - Hawthorne: LOC (地点)")
    print(f"    - California: LOC (地点)")
    print(f"    - Elon Musk: PER (人物)")

===== Hugging Face 更多 Pipeline 任务 =====

【任务2: 文本生成】
  Prompt: "Artificial intelligence is"
  生成: Artificial intelligence is a term that has been used for decades to describe

【任务3: 命名实体识别】
  文本: "Elon Musk founded SpaceX in Hawthorne, California."
  实体:
    - SpaceX: ORG (组织)
    - Hawthorne: LOC (地点)
    - California: LOC (地点)
    - Elon Musk: PER (人物)


### 4.3 使用模型和分词器

除了便捷的 `pipeline` API，还可以直接使用 `AutoModel` 和 `AutoTokenizer` 进行更灵活的操作：

In [9]:
from transformers import AutoTokenizer, AutoModel

print("===== 直接使用 AutoTokenizer + AutoModel =====")

# 加载预训练模型的分词器和模型
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 分词
text = "Natural language processing is fascinating"
inputs = tokenizer(text, return_tensors="pt")

print(f"\n【分词结果】")
print(f"  原始文本: \"{text}\"")
print(f"  Token IDs: {inputs['input_ids'].tolist()[0]}")
print(f"  Attention Mask: {inputs['attention_mask'].tolist()[0]}")
print(f"  Decoded: '{tokenizer.decode(inputs['input_ids'][0])}'")

# 模型推理
with torch.no_grad():
    outputs = model(**inputs)

print(f"\n【模型推理】")
print(f"  输出形状: {outputs.last_hidden_state.shape}")

# [CLS] 向量常用于分类任务的句子表示
cls_embedding = outputs.last_hidden_state[:, 0, :]  # 取第一个token [CLS]
print(f"  [CLS] 向量形状: {cls_embedding.shape}")
print(f"  [CLS] 向量前10维: {cls_embedding[0, :10].tolist()}")

===== 直接使用 AutoTokenizer + AutoModel =====

【分词结果】
  原始文本: "Natural language processing is fascinating"
  Token IDs: [225, 7123, 5090, 3635, 5826, 20873]
  Attention Mask: [1, 1, 1, 1, 1, 1]
  Decoded: 'natural language processing is fascinating'

【模型推理】
  输出形状: torch.Size([1, 6, 768])
  [CLS] 向量形状: torch.Size([1, 768])
  [CLS] 向量前10维: [ 0.0717,  0.0817, -0.0199, -0.0055,  0.1277,  0.1001,  0.0413, -0.1084,  0.0497,  0.0358]

---

## 五、Transformer 发展时间线

```
2017  Transformer          ← "Attention Is All You Need"
  │
2018  GPT-1               ← Decoder-only, 自回归
  │    BERT                ← Encoder-only, 双向, MLM+NSP
  │
2019  GPT-2               ← 大规模语言生成
  │    T5                  ← Encoder-Decoder, Text-to-Text
  │    XLNet/RoBERTa       ← BERT 改进
  │
2020  GPT-3               ← 175B 参数, Few-shot Learning
  │
2021  Codex               ← 代码生成
  │
2022  ChatGPT (GPT-3.5)   ← RLHF 对齐
  │    InstructGPT
  │
2023  GPT-4               ← 多模态
  │    LLaMA               ← 开源大模型
  │    BERT / T5 仍广泛用于下游任务
  │
2024  GPT-4o / Gemini / Claude 3  ← 多模态 Agent
```

---

## 六、本节知识图谱

```
                    自注意力机制
                   Q, K, V 三元组
            Attention(Q,K,V) = softmax(QK^T/√dk)V
                        │
              ┌─────────┼─────────┐
              ▼         ▼         ▼
       多头注意力   位置编码    残差+LayerNorm
       (Multi-Head)  (PE)      (Res+LN)
              │
              ▼
        Transformer 架构
        ┌──────┴──────┐
   Encoder-only  Decoder-only  Encoder-Decoder
       BERT         GPT          T5
       (双向)       (单向/自回归)  (生成+理解)
              │
              ▼
    Hugging Face Transformers
    pipeline API / AutoModel / AutoTokenizer
```

---

## 📝 练习题

### 练习 1：Self-Attention 手动计算（基础）

给定以下输入序列（3 个 token，每个 4 维）：

$$X = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \\ 1 & 1 & 0 & 0 \end{bmatrix}$$

假设 $W^Q = W^K = W^V = I$（单位矩阵），$d_k = 4$：

1. 计算 $Q = X, K = X, V = X$
2. 计算 $QK^T / \sqrt{4}$
3. 对每行做 softmax，得到注意力权重
4. 计算最终输出 $\text{softmax}(QK^T/\sqrt{4}) \cdot V$

提示：可参考本节的从零实现代码。

---

### 练习 2：Multi-Head Attention 扩展（进阶）

修改本节的多头注意力实现：

1. 将头数从 4 改为 8（调整 d_model 使得 d_k 仍为整数）
2. 实现因果掩码（Causal Mask），使模型只能看到当前位置及之前的内容
3. 输入句子 `"I love AI"`（假设已分词），可视化注意力权重矩阵

---

### 练习 3：Hugging Face 多任务实战（综合）

使用 Hugging Face `pipeline` API 完成以下任务：

1. **情感分析**：对 5 条电影评论进行正负面分类
2. **命名实体识别**：从一条新闻文本中提取所有人名、地名、机构名
3. **文本摘要**：对一篇长文章生成摘要（使用 `summarization` pipeline）
4. **问答**：给定上下文和问题，提取答案（使用 `question-answering` pipeline）

---

### 练习 4：完整 Transformer 编码器（挑战）

基于本节的组件，组装一个完整的 Transformer 编码器分类模型：

1. `Token Embedding` + `Positional Encoding`
2. $N=2$ 层 `TransformerEncoderLayer`
3. 取 `[CLS]` 位置的输出
4. 经过 `Linear(d_model → num_classes)` 得到分类 logits

用随机数据测试前向传播，验证输出形状为 `[batch, num_classes]`。

---

### 💡 拓展思考

1. 为什么 Transformer 能完全替代 RNN？它的核心优势是什么？
2. BERT 和 GPT 的架构差异如何影响它们各自擅长的任务？
3. 如果你要处理中文文本，应该选择哪个预训练模型？（提示：bert-base-chinese, text2vec-chinese 等）
4. 大语言模型（LLM）与传统的 BERT/GPT 有什么本质区别？